<a href="https://colab.research.google.com/github/ParamAhuja/DL_Notebooks/blob/main/HyperParamterTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# df

In [1]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/ParamAhuja/DL_Notebooks/refs/heads/main/datasets/diabetes.csv")

In [2]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.corr()["Outcome"]

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [4]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [5]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [6]:
X.shape, y.shape

((768, 8), (768,))

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [8]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [9]:
model = Sequential()
model.add(Dense(32, activation="relu", input_dim=8))
model.add(Dense(16, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer="Adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test))

Epoch 1/100


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.6571 - loss: 0.6382 - val_accuracy: 0.6883 - val_loss: 0.6008
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7077 - loss: 0.5855 - val_accuracy: 0.7208 - val_loss: 0.5701
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7092 - loss: 0.5497 - val_accuracy: 0.7403 - val_loss: 0.5459
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7105 - loss: 0.5262 - val_accuracy: 0.7727 - val_loss: 0.5256
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7349 - loss: 0.5001 - val_accuracy: 0.7597 - val_loss: 0.5120
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7364 - loss: 0.4921 - val_accuracy: 0.7662 - val_loss: 0.5017
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7661 - loss: 0.4742 - val_accuracy: 0.7662 - val_loss: 0.4931
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7669 - loss: 0.4793 - val_accuracy: 0.7922 - val_loss:

# keras-tuner: hp.Choice
hyperparamter tuning for best optimizer

In [17]:
!pip install -q keras-tuner

In [18]:
# import keras_tuner as kt
import kerastuner as kt
# for some reason there are 2 modules, 1 might be deprecated

In [19]:
def build_model(hp):
  """conventional name for function is build_model
  paramter: hp (hyperparameter) object from keras tuner
  returns : model object optimized
  """
  # lets tune the optimizer
  model = Sequential()
  model.add(Dense(32, activation="relu", input_dim=8))
  model.add(Dense(16, activation="relu"))
  model.add(Dense(1, activation="sigmoid"))

  optimizer =hp.Choice("optimizer", ["adam", "rmsprop", "adagrad", "sgd", "adadelta"])
  model.compile(optimizer = optimizer,
                loss="binary_crossentropy",
                metrics=["accuracy"])
  return model

In [20]:
tuner = kt.RandomSearch(
    build_model,
    objective = "val_accuracy",
    max_trials = 5,
    directory = "mydir",
    project_name = "myproject1"
    )

In [21]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7077922224998474

Best val_accuracy So Far: 0.7857142686843872
Total elapsed time: 00h 00m 24s


In [22]:
tuner.results_summary()

Results summary
Results in mydir/myproject1
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.7857142686843872

Trial 1 summary
Hyperparameters:
optimizer: adam
Score: 0.7467532753944397

Trial 4 summary
Hyperparameters:
optimizer: sgd
Score: 0.7077922224998474

Trial 0 summary
Hyperparameters:
optimizer: adadelta
Score: 0.551948070526123

Trial 2 summary
Hyperparameters:
optimizer: adagrad
Score: 0.4610389471054077


In [23]:
tuner.get_best_hyperparameters()[0]

In [24]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [25]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [26]:
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test), initial_epoch = 5)
# initial epochs to start from where it stopped in tuner (5 done)

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.7759 - loss: 0.5183 - val_accuracy: 0.7987 - val_loss: 0.4912
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7682 - loss: 0.5049 - val_accuracy: 0.8052 - val_loss: 0.4742
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7853 - loss: 0.4862 - val_accuracy: 0.7987 - val_loss: 0.4667
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7957 - loss: 0.4554 - val_accuracy: 0.8052 - val_loss: 0.4602
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7908 - loss: 0.4548 - val_accuracy: 0.8117 - val_loss: 0.4579
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7735 - loss: 0.4541 - val_accuracy: 0.8182 - val_loss: 0.4548
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7464 - loss: 0.4741 - val_accuracy: 0.8182 - val_loss: 0.4542
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7451 - loss: 0.4865 - val_accuracy

# keras-tuner: hp.Int
hyperparamter tuning for number of nodes

In [27]:
def build_model(hp):
  model = Sequential()
  units = hp.Int("units", min_value=8, max_value=128, step=8)
  model.add(Dense(units=units, activation="relu", input_dim=8))
  model.add(Dense(1, activation="sigmoid"))

  model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
  return model

In [28]:
tuner = kt.RandomSearch(
    build_model,
    objective = "val_accuracy",
    max_trials = 5,
    directory = "mydir",
    project_name = "myproject2"
)

**Analyze the directory**

In [29]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 05s]
val_accuracy: 0.7922077775001526

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 17s


In [30]:
tuner.results_summary()

Results summary
Results in mydir/myproject2
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 1 summary
Hyperparameters:
units: 96
Score: 0.8051947951316833

Trial 4 summary
Hyperparameters:
units: 80
Score: 0.7922077775001526

Trial 0 summary
Hyperparameters:
units: 48
Score: 0.7857142686843872

Trial 2 summary
Hyperparameters:
units: 8
Score: 0.6428571343421936

Trial 3 summary
Hyperparameters:
units: 16
Score: 0.6363636255264282


In [31]:
tuner.get_best_hyperparameters()[0].values

{'units': 96}

In [32]:
model = tuner.get_best_models(num_models = 1)[0]

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [33]:
model.fit(X_train, y_train, batch_size=32, epochs=100, verbose=1, validation_data = (X_test, y_test), initial_epoch = 5)

Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.7477 - loss: 0.5325 - val_accuracy: 0.8052 - val_loss: 0.4910
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7884 - loss: 0.4822 - val_accuracy: 0.7987 - val_loss: 0.4788
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7742 - loss: 0.4973 - val_accuracy: 0.7922 - val_loss: 0.4713
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7736 - loss: 0.4846 - val_accuracy: 0.7987 - val_loss: 0.4666
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7899 - loss: 0.4559 - val_accuracy: 0.8052 - val_loss: 0.4647
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7780 - loss: 0.4487 - val_accuracy: 0.8247 - val_loss: 0.4645
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7701 - loss: 0.4696 - val_accuracy: 0.8247 - val_loss: 0.4634
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7980 - loss: 0.4560 - val_accuracy: 0.

# keras tuner
tuning no. of layers